# Toy Transformer Setup (V=5, L=4)

This notebook records the simplified transformer attention setup using the **same conventions as most transformer papers**:

- Token representations are **row vectors**.
- Matrices multiply on the **right**.
- Attention output is computed as **H = P V**.


## Dimensions

- Vocabulary size: **V = 5**
- Context length: **L = 4**
- Embedding dimension: **d = V + L = 9**

Token representations:

$$
x \in \mathbb{R}^{1 \times d}
$$


## Basis vectors

Standard basis vectors are **row vectors**:

$$
e_i \in \mathbb{R}^{1 \times d}
$$

Example:

$$
e_1 = [1,0,0,0,0,0,0,0,0]
$$


## Token and position representation

Token `t` at position `p` is represented by

$$
x(t,p) = e_t + e_{V+p}
$$

Thus each vector has **two non‑zero coordinates**:

- one for the token
- one for the position


## Stacked input matrix

Stacking token vectors gives

$$
X \in \mathbb{R}^{L \times d}
$$

Each row corresponds to one position in the sequence.


## Linear projections

$$
Q = X W_Q
$$

$$
K = X W_K
$$

$$
V = X W_V
$$

with

$$
W_Q, W_K, W_V \in \mathbb{R}^{d \times d}
$$


## Attention scores

$$
S = Q K^T
$$

Shape:

$$
S \in \mathbb{R}^{L \times L}
$$

Entry $S_{ij}$ is the score between query position $i$ and key position $j$.


## Attention probabilities

Apply row‑wise softmax:

$$
P = \text{softmax}(S)
$$

Each row sums to 1.


## Attention output

$$
H = P V
$$

Shapes:

- $P \in \mathbb{R}^{L \times L}$
- $V \in \mathbb{R}^{L \times d_v}$
- $H \in \mathbb{R}^{L \times d_v}$

Row $i$:

$$
h_i = \sum_{j=1}^{L} P_{ij} v_j
$$


## Final output (optional)

If predicting a token distribution:

$$
z = h_L W_O
$$

$$
p = \text{softmax}(z)
$$

where

$$
W_O \in \mathbb{R}^{d_v \times V}
$$


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Parameters from context
V = 5           # Vocabulary size
L = 4           # Context length
d_model = V + L # Dimension (9)

In [ ]:
def get_input_matrix(token_indices):
    """
    Constructs X in R^{L x d} using torch operations
    """
    # Ensure tokens are a tensor
    tokens = torch.tensor(token_indices)
    
    # Content: e_t
    content_eb = F.one_hot(tokens, num_classes=V).float() # (L, V)
    
    # Position: e_{V+p}
    pos_indices = torch.arange(L)
    pos_eb = F.one_hot(pos_indices, num_classes=L).float() # (L, L)
    
    # Concatenate to form x(t,p) = e_t + e_{V+p}
    X = torch.cat([content_eb, pos_eb], dim=-1) # (L, V+L)
    return X

# Example Sequence: [0, 2, 1, 4]
X = get_input_matrix([0, 2, 1, 4])

In [ ]:
# Standard transformer conventions
# We use bias=False to strictly follow the notation Q = XW_Q
W_Q = torch.nn.Linear(d_model, d_model, bias=False)
W_K = torch.nn.Linear(d_model, d_model, bias=False)
W_V = torch.nn.Linear(d_model, V, bias=False) # d_v = V

# Initialize as identity for the 'Probabilistic Pointer' behavior
with torch.no_grad():
    W_Q.weight.copy_(torch.eye(d_model))
    W_K.weight.copy_(torch.eye(d_model))
    # Identity for the first V components
    W_V.weight.copy_(torch.eye(d_model)[:V, :]) 

Q = W_Q(X)
K = W_K(X)
V_mat = W_V(X)

In [ ]:
# S = Q K^T
scores = torch.matmul(Q, K.transpose(-2, -1))

# Causal Masking
mask = torch.tril(torch.ones(L, L))
scores = scores.masked_fill(mask == 0, float('-inf'))

# P = softmax(S)
P = F.softmax(scores, dim=-1)

# H = P V
H = torch.matmul(P, V_mat)

## Relationship Matrix R

The score matrix can be factored through a **relationship matrix** $R \in \mathbb{R}^{d \times d}$ that captures pairwise attention affinities between **embedding directions** (tokens/positions), independently of any specific input sequence.

Starting from $S = QK^T$ and substituting $Q = XW_Q$, $K = XW_K$:

$$
S = (XW_Q)(XW_K)^T = X \underbrace{(W_Q W_K^T)}_{R} X^T
$$

So:

$$
\boxed{R = W_Q W_K^T \in \mathbb{R}^{d \times d}, \qquad S_{\text{alt}} = X R X^T \in \mathbb{R}^{L \times L}}
$$

**Interpretation:** Entry $R_{ab}$ measures how much embedding direction $e_a$ (as a query) attends to embedding direction $e_b$ (as a key). Because the token representations $x(t,p) = e_t + e_{V+p}$ are sums of basis vectors, the score between position $i$ and position $j$ decomposes as a sum of $R$ entries over the active (token, position) pairs:

$$
S_{ij} = x_i R x_j^T = R_{t_i, t_j} + R_{t_i, V+p_j} + R_{V+p_i, t_j} + R_{V+p_i, V+p_j}
$$

This lets us read off **which token-to-token, token-to-position, and position-to-position interactions** drive each attention score — without needing to enumerate sequences.

In [ ]:
# R = W_Q W_K^T in the mathematical (row-vector) convention.
#
# PyTorch's nn.Linear stores weights transposed relative to the mathematical matrix:
#   Q = W_Q(X)  computes  X @ W_Q.weight.T   (i.e. mathematical W_Q = W_Q.weight.T)
#   K = W_K(X)  computes  X @ W_K.weight.T   (i.e. mathematical W_K = W_K.weight.T)
#
# So:  R = W_Q_math @ W_K_math^T
#         = W_Q.weight.T @ W_K.weight

R = W_Q.weight.T @ W_K.weight          # shape (d_model, d_model)

# Alternative score computation: S_alt = X R X^T
scores_alt = X @ R @ X.T               # shape (L, L)

# Verify equivalence with the original score matrix (before masking)
scores_orig = torch.matmul(Q, K.transpose(-2, -1))
print("Max absolute difference between scores and scores_alt:",
      (scores_alt - scores_orig).abs().max().item())
print("\nR (relationship matrix):\n", R.detach().numpy().round(3))
print("\nscores_alt:\n", scores_alt.detach().numpy().round(4))

## Visualizing R and S_alt

The heatmaps below show:
- **Left**: $R \in \mathbb{R}^{d \times d}$ — attention affinity between every pair of embedding dimensions. Axes span token dimensions $0\ldots V{-}1$ then position dimensions $V\ldots V{+}L{-}1$.
- **Right**: $S_{\text{alt}} \in \mathbb{R}^{L \times L}$ — pre-softmax scores for the current sequence, computed via $X R X^T$. Compare with the masked version used in attention.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

R_np = R.detach().numpy()
S_np = scores_alt.detach().numpy()

# Axis labels for R: t0..t4 then p0..p3
dim_labels = [f"t{i}" for i in range(V)] + [f"p{i}" for i in range(L)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- R heatmap ---
ax = axes[0]
R_absmax = np.abs(R_np).max()
im = ax.imshow(R_np, cmap="RdBu_r", vmin=-R_absmax, vmax=R_absmax)
ax.set_title(r"Relationship matrix $R = W_Q W_K^T$  $(d \times d)$", fontsize=12)
ax.set_xlabel("Key dimension (embedding index)")
ax.set_ylabel("Query dimension (embedding index)")
ax.set_xticks(range(d_model)); ax.set_xticklabels(dim_labels, rotation=45, ha="right")
ax.set_yticks(range(d_model)); ax.set_yticklabels(dim_labels)
# Draw dividing lines between token and position blocks
for val in [V - 0.5]:
    ax.axhline(val, color="black", linewidth=1.5, linestyle="--")
    ax.axvline(val, color="black", linewidth=1.5, linestyle="--")
plt.colorbar(im, ax=ax, fraction=0.046)
# Annotate cells with values
for i in range(d_model):
    for j in range(d_model):
        ax.text(j, i, f"{R_np[i,j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if abs(R_np[i,j]) > 0.5 * R_absmax else "black")

# --- S_alt heatmap ---
ax = axes[1]
im2 = ax.imshow(S_np, cmap="viridis")
ax.set_title(r"Score matrix $S_{\mathrm{alt}} = X R X^T$  $(L \times L)$", fontsize=12)
ax.set_xlabel("Key position j")
ax.set_ylabel("Query position i")
ax.set_xticks(range(L)); ax.set_xticklabels([f"pos {j}" for j in range(L)])
ax.set_yticks(range(L)); ax.set_yticklabels([f"pos {i}" for i in range(L)])
plt.colorbar(im2, ax=ax, fraction=0.046)
for i in range(L):
    for j in range(L):
        ax.text(j, i, f"{S_np[i,j]:.2f}", ha="center", va="center", fontsize=10, color="white")

plt.tight_layout()
plt.show()

In [ ]:
# The finite-state machine interpretation
last_token_probs = P[-1, :] # Probability distribution over previous positions

print(f"Attention weights for the last query: {last_token_probs.detach().numpy()}")
print(f"Resulting distribution over vocabulary: {H[-1, :].detach().numpy()}")